# Chapter 7 &mdash; Reversal of a DFA Yields an NFA

**Concept 11 of the Chapter 7 decomposition:** *Reversal of a DFA Yields an NFA*

Flip every arrow: final states become the initial set, the old start becomes the sole final state.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter7/Concept-Reversal-Yields-NFA/Concept-Reversal-Yields-NFA.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.AnimateNFA     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


Reversing a DFA is mechanical: **flip every arrow**.

* $F$ becomes the **initial set** $Q_0$ &mdash; and since $F$ may have many states, the
  result is genuinely an **NFA**, not a DFA;
* the old $q_0$ becomes the **sole final state**;
* $\delta^R(q',a) = \{q : \delta(q,a)=q'\}$, which is a set because several states can
  share a target.

So reversal is the first operation that *forces* nondeterminism. It also proves that
regular languages are **closed under reversal** &mdash; the fact Chapter 4 used to settle
$L_{if}$.

## 2. Definitions

### A DFA and its reversal

In [ ]:
D = md2mc('''DFA
I  : 0 -> A
I  : 1 -> I
A  : 0 -> F
A  : 1 -> I
F  : 0 -> F
F  : 1 -> I
''')
R = rev_dfa(D)
print("D : q0 = %r, F = %s" % (D["q0"], sorted(D["F"])))
print("R : Q0 = %s, F = %s" % (sorted(R["Q0"]), sorted(R["F"])))

### Reversal, by hand, to see the set-valued $\delta$

In [ ]:
def reverse(D):
    Dl = {}
    for (q, a), t in D["Delta"].items():
        Dl.setdefault((t, a), set()).add(q)
    return mk_nfa(D["Q"], D["Sigma"], Dl, set(D["F"]), {D["q0"]})

## 3. Tests

The reversed machine accepts exactly the reversed strings.

In [ ]:
from itertools import product
strs = [''.join(p) for k in range(11) for p in product('01', repeat=k)]
bad = [s for s in strs if accepts_nfa(R, s) != accepts_dfa(D, s[::-1])]
print("mismatches :", len(bad))
assert not bad
print("L(R) = { w^R : w in L(D) }, verified on all %d strings" % len(strs))

The old $F$ becomes the start **set** &mdash; nondeterminism is forced.

In [ ]:
print("|F| of the original  :", len(D["F"]))
print("|Q0| of the reversal :", len(R["Q0"]))
assert R["Q0"] == D["F"]
print("\nA DFA may have many final states, so the reversal may have many start states,")
print("and a machine with several start states is by definition an NFA.")

$\delta^R$ really is set-valued where two states shared a target.

In [ ]:
mine = reverse(D)
multi = [(k, sorted(v)) for k, v in mine["Delta"].items() if len(v) > 1]
print("set-valued entries :", multi)
assert multi, "some target must have had two predecessors"

Our hand reversal agrees with `rev_dfa`.

In [ ]:
assert all(accepts_nfa(mine, s) == accepts_nfa(R, s) for s in strs)
print("hand-written reverse agrees with rev_dfa everywhere")

Closure under reversal, which Chapter 4 used on $L_{if}$.

In [ ]:
RR = nfa2dfa(rev_dfa(nfa2dfa(rev_dfa(D))))
print("reverse twice returns the original language? ", langeq_dfa(RR, D))
assert langeq_dfa(RR, D)

## 4. Animation

The reversed machine: several start states, one final state, every arrow flipped.

*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimateNFA import *
AnimateNFA(R, FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Reverse a DFA with exactly one final state. Is the result deterministic?
2. Prove $L^{RR} = L$ from the construction.
3. Which chapter-4 proof depended on closure under reversal?

In [ ]:
# Your work for the exercises above.